#**NYC MTA bus speeds**

This is an analysis of NYC bus speeds by route using MTA open data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
url = "https://data.ny.gov/api/views/cudb-vcni/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(url)

In [ ]:
df.head()
df.shape
df.dtypes
df.isnull().sum()

# **Which routes have the slowest average speed overall?**

In [ ]:
route_speeds = df.groupby ("route_id")["average_speed"].mean()
route_speeds.sort_values().head(10)
regular_routes = df[df["route_id"].str.contains("SHTL") == False]
route_speeds_clean = regular_routes.groupby("route_id")["average_speed"].mean()
slowest = route_speeds_clean.sort_values().head(10)
slowest.plot(kind="bar")
plt.title("10 Slowest NYC Bus Routes by Average Speed")
plt.xlabel("Route")
plt.ylabel("Average Speed (mph)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# **Does speed change by Borough?**

In [ ]:
borough_speeds = regular_routes.groupby("borough")["average_speed"].mean()
borough_speeds.sort_values().plot(kind="bar")
plt.title("Average Speed By Borough")
plt.xlabel("Borough")
plt.ylabel("Average Speed (mph)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# **Visualizations**

In [ ]:
import geopandas as gpd

url_routes = "https://data.ny.gov/resource/bzwk-3hb4.geojson?$where=in_effect='true'&$limit=5000"
routes = gpd.read_file(url_routes)
print(routes.shape)
print(df.shape)
print("Routes dataset sample:",routes["route_id"].head(20).tolist())
print("Speed dataset sample:", regular_routes["route_id"].head(20).tolist())
print("Unique routes in map data:", routes["route_id"].nunique())
print("Unique routes in speed data:", regular_routes["route_id"].nunique())

routes_one = routes.drop_duplicates(subset="route_id")
print(routes_one.shape)

merged = routes_one.merge(route_speeds_clean, on="route_id", how="inner")
print(merged.shape)
print(merged.columns.tolist())

merged["speed_normalized"] = (merged["average_speed"] - merged["average_speed"].min()) / (merged["average_speed"].max() - merged["average_speed"].min())
# Take each speed value, subtract the slowest speed so the minimum becomes zero, then divide by the total range so the maximum becomes one. Every route ends up somewhere between 0 and 1
# for every row, do (this speed - slowest speed) / total range

# **NYC Map**

In [ ]:
import folium
import branca.colormap as cm

m = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles="CartoDB positron",
    min_zoom=10,
    max_bounds=True
)

colormap = cm.LinearColormap(
    colors=["red", "yellow", "green"],
    vmin=merged["average_speed"].min(),
    vmax=merged["average_speed"].max(),
    caption="Average Bus Speed (mph)"
)

for _, row in merged.iterrows():
    feature = {
        "type": "Feature",
        "geometry": row["geometry"].__geo_interface__,
        "properties": {
            "route_id": row["route_id"],
            "route_long_name": row["route_long_name"],
            "average_speed": round(row["average_speed"], 2)
        }
    }
    folium.GeoJson(
        feature,
        style_function=lambda x, speed=row["average_speed"]: {
            "color": colormap(speed),
            "weight": 2,
            "opacity": 0.7
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["route_id", "route_long_name", "average_speed"],
            aliases=["Route:", "Name:", "Avg Speed (mph):"],
            localize=True
        )
    ).add_to(m)

# for every route, package its geometry and data into the format folium expects,
# style it with a color based on its speed, attach a tooltip so hovering shows
# the route info, and add it to the map.

colormap.add_to(m)
m.fit_bounds([[40.4774, -74.2591], [40.9176, -73.7004]])
m

In [ ]:
df["month"] = pd.to_datetime(df["month"])
print(df["month"].dtype)
print(df["month"].head(5))

df["year"] = df["month"].dt.year
df["year"].value_counts().sort_index()

regular_routes = regular_routes.copy()
regular_routes["month"] = pd.to_datetime(regular_routes["month"])
regular_routes["year"] = regular_routes["month"].dt.year

# **Interactive Graphs**

In [ ]:
import plotly.express as px

slowest_df = slowest.reset_index()
slowest_df.columns = ["route_id", "average_speed"]

fig = px.bar(
    slowest_df,
    x="route_id",
    y="average_speed",
    color="average_speed",
    color_continuous_scale=["darkred", "red", "lightsalmon"],
    title="10 Slowest NYC Bus Routes by Average Speed",
    labels={"route_id": "Route" , "average_speed": "Average Speed (mph) "}
)

fig.update_traces(hovertemplate="<b>Route %{x}</b><br>Avg Speed: %{y:.2f} mph<extra></extra>")
fig.show()

In [ ]:
borough_speeds_df = borough_speeds.reset_index()
borough_speeds_df.columns = ["borough", "average_speed"]

fig = px.bar(
    borough_speeds_df,
    x="borough",
    y="average_speed",
    color="average_speed",
    color_continuous_scale=["red", "yellow", "green"],
    title="Average Speed By Borough",
    labels={"borough": "Borough", "average_speed": "Average Speed (mph)"}
)

fig.update_traces(hovertemplate="<b>Borough %{x}</b><br>Avg Speed: %{y:.2f} mph<extra></extra>")
fig.update_layout(xaxis={"categoryorder": "total ascending"})
fig.show()

# **COVID Analysis**

In [ ]:
yearly_speed = regular_routes.groupby("year")["average_speed"].mean().reset_index()

fig = px.line(
    yearly_speed,
    x="year",
    y="average_speed",
    title="NYC MTA Bus Speed Over Time (2015–2026)",
    labels={"year": "Year", "average_speed": "Average Speed (mph)"},
    markers=True
)

fig.add_vline(
    x=2020,
    line_dash="dash",
    line_color="black",
    annotation_text="COVID-19",
    annotation_position="top right"
)

fig.add_vline(
    x=2019,
    line_dash="dash",
    line_color="red",
    annotation_text="Bus Camera Enforcement Begins",
    annotation_position="top left"
)

fig.add_vline(
    x=2022,
    line_dash="dash",
    line_color="green",
    annotation_text="Hayden AI Deployment",
    annotation_position="top right"
)

fig.update_traces(hovertemplate="<b>%{x}</b><br>Avg Speed: %{y:.2f} mph<extra></extra>")

fig.show()

# **B44 Speed Average Analysis**

In [ ]:
b44_speed = route_speeds_clean.loc["B44"]
b44_annual_speed = regular_routes[regular_routes["route_id"] == "B44"].groupby("year")["average_speed"].mean().reset_index()

print(f"Overall average speed for B44 route: {b44_speed:.2f} mph")
print("\nAnnual average speed for B44 route:")
print(b44_annual_speed)

fig = px.line(
    b44_annual_speed,
    x="year",
    y="average_speed",
    title="B44 Speed Over Time (2015–2026)",
    labels={"year": "Year", "average_speed": "Average Speed (mph)"},
    markers=True
)


fig.add_vline(
    x=2022,
    line_dash="dash",
    line_color="green",
    annotation_text="Hayden AI Deployment",
    annotation_position="top right"
)

fig.add_scatter(
    x=yearly_speed["year"],
    y=yearly_speed["average_speed"],
    name="Citywide average",
    line=dict(dash="dot", color="gray")
)

fig.update_traces(hovertemplate="<b>%{x}</b><br>Avg Speed: %{y:.2f} mph<extra></extra>")

fig.show()

# **Peak and Off-Peak Analysis**

In [ ]:
peak_analysis = regular_routes.groupby(["route_id", "period"])["average_speed"].mean().reset_index()
peak = peak_analysis[peak_analysis["period"] == "Peak"]
off_peak = peak_analysis[peak_analysis["period"] == "Off peak"]

# get the 10 slowest route names during peak
slowest_peak_routes = peak.sort_values("average_speed").head(10)["route_id"].tolist()
# filter peak_analysis to keep only those 10 routes, both periods
peak_comparison = peak_analysis[peak_analysis["route_id"].isin(slowest_peak_routes)]

fig = px.bar(
    peak_comparison,
    x="route_id",
    y="average_speed",
    color="period",
    barmode="group",
    title="Rush hour vs off-peak speed for the 10 slowest peak-hour routes",
    labels={"route_id": "Route", "average_speed": "Average Speed (mph)", "period": "Period"},
    color_discrete_map={"Peak": "purple", "Off-Peak": "pink"}
)

fig.update_traces(hovertemplate="<b>Route %{x}</b><br>Avg Speed: %{y:.2f} mph<extra></extra>")
fig.update_layout(xaxis={"categoryorder": "total ascending"})
fig.show()
